In [1]:

import pandas as pd
import requests
import sqlite3

from bs4 import BeautifulSoup
from datetime import datetime

url = "https://en.wikipedia.org/wiki/List_of_countries_by_GDP_(nominal)"

headers = {
    "User-Agent": "ETLLearningProject/1.0 (educational project; contact: your-email@example.com)"
}

db_name = "World_Economies.db"
table_name = "Countries_by_GDP"
csv_path = "Countries_by_GDP.csv"
log_file = "etl_project_log.txt"

table_attribs = ["Country", "GDP_USD_millions"]


In [2]:

def extract(url, table_attribs):
    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    tables = soup.find_all("table")

    # Confirm the IMF GDP table is still at index 2.
    table = tables[2]
    rows = table.find_all("tr")

    records = []

    for row in rows:
        cols = row.find_all("td")

        if len(cols) < 3:
            continue

        country_cell = cols[0]
        gdp_cell = cols[1]

        # Exclude aggregate rows without country links.
        if country_cell.find("a", href=True) is None:
            continue

        country = country_cell.get_text(" ", strip=True)
        gdp = gdp_cell.get_text(" ", strip=True)

        if gdp in ("—", "", "N/A"):
            continue

        records.append({
            "Country": country,
            "GDP_USD_millions": gdp
        })

    df = pd.DataFrame(records, columns=table_attribs)

    return df


In [3]:
extracted_data = extract(url, table_attribs)

print(extracted_data.head())
print("Records extracted:", len(extracted_data))

          Country GDP_USD_millions
0   United States       32,642,373
1   China [ n 1 ]       21,546,403
2         Germany        5,471,299
3           Japan        4,341,986
4  United Kingdom        4,268,446
Records extracted: 221


In [4]:

def transform(df):
    df = df.copy()

    df["Country"] = (
        df["Country"]
        .str.replace(r"\s*\[.*?\]", "", regex=True)
        .str.strip()
    )

    df = df[df["Country"] != "World"]

    df["GDP_USD_millions"] = (
        df["GDP_USD_millions"]
        .str.replace(",", "", regex=False)
    )

    df["GDP_USD_millions"] = pd.to_numeric(
        df["GDP_USD_millions"],
        errors="coerce"
    )

    df = df.dropna(subset=["GDP_USD_millions"])

    df["GDP_USD_millions"] = (
        df["GDP_USD_millions"] / 1000
    ).round(2)

    df = df.rename(columns={
        "GDP_USD_millions": "GDP_USD_billions"
    })

    return df


In [5]:
transformed_data = transform(extracted_data)

print(transformed_data.head())
print(transformed_data.dtypes)

          Country  GDP_USD_billions
0   United States          32642.37
1           China          21546.40
2         Germany           5471.30
3           Japan           4341.99
4  United Kingdom           4268.45
Country                 str
GDP_USD_billions    float64
dtype: object


In [6]:

def load_to_csv(df, csv_path):
    df.to_csv(csv_path, index=False)


In [7]:

def load_to_db(df, conn, table_name):
    df.to_sql(
        table_name,
        conn,
        if_exists="replace",
        index=False
    )


In [8]:

def run_query(query, conn):
    result = pd.read_sql_query(query, conn)
    print(result)
    return result


In [9]:

def log_progress(message):
    timestamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    with open(log_file, "a") as file:
        file.write(f"{timestamp} : {message}\n")


In [10]:

def main():
    log_progress("ETL process started")

    # Extract
    extracted_data = extract(url, table_attribs)
    log_progress("Data extraction complete")

    # Transform
    transformed_data = transform(extracted_data)
    log_progress("Data transformation complete")

    # Load to CSV
    load_to_csv(transformed_data, csv_path)
    log_progress("Data saved to CSV")

    # Load to SQLite and query
    with sqlite3.connect(db_name) as conn:
        load_to_db(transformed_data, conn, table_name)
        log_progress("Data loaded to SQLite")

        query = f"""
        SELECT *
        FROM {table_name}
        WHERE GDP_USD_billions >= 100
        """

        result = run_query(query, conn)
        log_progress("SQL query completed")

    log_progress("ETL process completed")

    return transformed_data, result


if __name__ == "__main__":
    final_data, query_result = main()


           Country  GDP_USD_billions
0    United States          32642.37
1            China          21546.40
2          Germany           5471.30
3            Japan           4341.99
4   United Kingdom           4268.45
..             ...               ...
76       Venezuela            111.30
77      Luxembourg            110.42
78      Costa Rica            109.93
79       Lithuania            105.91
80         Belarus            102.04

[81 rows x 2 columns]


In [11]:

print("Total records:", len(final_data))

print("Missing values:")
print(final_data.isnull().sum())

print("Duplicate countries:")
print(final_data["Country"].duplicated().sum())

print("Column types:")
print(final_data.dtypes)

print("Countries above $100 billion:")
print(len(query_result))


Total records: 191
Missing values:
Country             0
GDP_USD_billions    0
dtype: int64
Duplicate countries:
0
Column types:
Country                 str
GDP_USD_billions    float64
dtype: object
Countries above $100 billion:
81


In [13]:
print("Extracted:", len(extracted_data))
print("Transformed:", len(final_data))
print("Removed:", len(extracted_data) - len(final_data))

Extracted: 221
Transformed: 191
Removed: 30


In [12]:
with open(log_file, "r") as file:
    print(file.read())

2026-11-14 21:35:44 : Data extraction complete.
2026-11-14 21:35:44 : Data transformation complete.
2026-11-14 21:35:44 : Data loaded successfully.
2026-Nov-14-21:42:31,ETL Job Started
2026-Nov-14-21:42:31,Extract phase Started
2026-11-14 21:51:45 : ETL process started
2026-11-14 21:51:45 : Data extraction complete
2026-11-14 21:51:45 : Data transformation complete
2026-11-14 21:51:45 : Data saved to CSV
2026-11-14 21:51:45 : Data loaded to SQLite
2026-11-14 21:51:45 : SQL query completed
2026-11-14 21:51:45 : ETL process completed
2026-11-14 21:53:43 : ETL process started
2026-11-14 21:53:43 : Data extraction complete
2026-11-14 21:53:43 : Data transformation complete
2026-11-14 21:53:43 : Data saved to CSV
2026-11-14 21:53:43 : Data loaded to SQLite
2026-11-14 21:53:43 : SQL query completed
2026-11-14 21:53:43 : ETL process completed



In [14]:
final_data.to_csv("Countries_by_GDP.csv", index=False)

print("CSV file saved successfully!")

CSV file saved successfully!


In [15]:
import os

print(os.path.abspath("Countries_by_GDP.csv"))

/Users/priscillagudisa/Countries_by_GDP.csv
